# 🚕 What Predicts the Speed of a RoboTaxi Crash?
### Multiple Linear Regression of Autonomous Vehicle Crash Speeds
**PROG8431 · Problem Analysis Workshop C · Team 3**

| Name | Student ID |
| :--- | :--- |
| Sultan Atanda | 9114837 |
| Ricardo Mohammed | 7500382 |
| Alamir Ibrahim | 9104960 |

> **Data:** NHTSA Standing General Order 2021-01 crash reports (augmented in Workshop C)<br>

> **Question:** what factors predict how fast a RoboTaxi is moving when it crashes?<br>

> **Repo Link:** https://github.com/kinghighshow/-WorkshopC_Team_3.git

**Roadmap:** ① Augment & transform the data → ② Split train/test & validate → ③ Choose the dependent variable → ④ Build & compare 2 SLR models → ⑤ Build & refine the MLR model → ⑥ Evaluate & discuss

### Start by clearing all output and workspace
This ensures a reproducible run from a clean state every time the notebook is executed top-to-bottom.

In [1]:
# @title
from IPython.display import clear_output
clear_output()
%reset -f

### Import necessary packages and set constants
We import the standard data stack plus the statistical tools needed for this workshop: `scipy.stats.boxcox` for the Box-Cox transformation, `scipy.stats` for skewness diagnostics, and `statsmodels` for OLS regression with full summary tables. Constants are defined up top so every later section references the same names.

In [2]:
# @title
%pip install -q pandas numpy scipy scikit-learn plotly geonamescache astral statsmodels

Note: you may need to restart the kernel to use updated packages.


# 📂 Import the data

In [3]:
# @title
import warnings
from pathlib import Path
from urllib.request import urlretrieve
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import geonamescache
from astral import Observer
from astral.sun import elevation
from scipy.stats import boxcox, skew, ks_2samp, chi2_contingency
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 60)

RAW_FILE = "SGO-2021-01_Incident_Reports_ADS.csv"
RAW_URL = "https://raw.githubusercontent.com/kinghighshow/-WorkshopC_Team_3/main/" + RAW_FILE
STUDY_FILE = "AV_Incidents_Study.csv"
SEED = 6174

### 1.1 Load the raw data

In [4]:
# @title
if not Path(RAW_FILE).exists():
    urlretrieve(RAW_URL, RAW_FILE)
    print(f"Downloaded {RAW_FILE} from GitHub")

raw = pd.read_csv(RAW_FILE, low_memory=False)
print(f"Raw data: {raw.shape[0]:,} rows × {raw.shape[1]} columns")

Raw data: 1,365 rows × 116 columns


### 1.2 Clean
- Keep only the latest version of each report
- Keep one row per crash
- Drop columns with no use: IDs, hidden personal info, free text, single-value columns
- Remove crashes where the AV's speed was 0 (stopped or parked AVs that were hit)

In [5]:
# @title
class IncidentCleaner:
    """Removes duplicate reports and columns that carry no usable information."""

    ID_COLS = ["Report ID", "Report Version", "Report Type", "Report Month", "Report Year",
               "Report Submission Date", "VIN", "VIN Decoded", "Serial Number",
               "Same Vehicle ID", "Same Incident ID", "Automation Feature Version",
               "Operating Entity", "Narrative"]
    PII_COLS = ["Latitude", "Longitude", "Address", "Zip Code"]

    def __init__(self, df):
        self.df = df.copy()
        self.log = [("Raw file", *df.shape)]

    def _record(self, step):
        self.log.append((step, *self.df.shape))

    def dedupe(self):
        """Keeps the latest version of each report and one row per crash."""
        self.df = (self.df.sort_values("Report Version")
                          .drop_duplicates("Report ID", keep="last"))
        self._record("Latest version per Report ID")
        self.df = self.df.drop_duplicates("Same Incident ID", keep="first")
        self._record("One row per crash (Same Incident ID)")
        return self

    def drop_columns(self):
        """Drops identifier, PII, confidential and single-value columns."""
        blank = self.df.replace(r"^\s*$", np.nan, regex=True)
        single_value = [c for c in blank.columns if blank[c].nunique(dropna=False) <= 1]
        drop = set(self.ID_COLS + self.PII_COLS + single_value)
        drop |= {c for c in self.df.columns if c.endswith("CBI") or c.endswith("CBI?")}
        self.df = self.df.drop(columns=[c for c in drop if c in self.df.columns])
        self._record("Drop ID / PII / constant columns")
        return self

    def drop_zero_speed(self):
        """Removes crashes where the AV was not moving."""
        self.df = self.df[self.df["SV Precrash Speed (MPH)"] > 0]
        self._record("Remove zero-speed crashes")
        return self

    def summary(self):
        """Returns the row and column count after each cleaning step."""
        return pd.DataFrame(self.log, columns=["Step", "Rows", "Columns"]).set_index("Step")


cleaner = IncidentCleaner(raw).dedupe().drop_columns().drop_zero_speed()
cleaner.summary()

,Rows,Columns
Step,,
Raw file,1365,116
Latest version per Report ID,1324,116
One row per crash (Same Incident ID),1323,116
Drop ID / PII / constant columns,1323,68
Remove zero-speed crashes,630,68


### 1.3 Add outside data
The crash file hides locations and has no lighting info, so we added:

| Source | New variables |
|---|---|
| **GeoNames** city database (via `geonamescache`) | City population, latitude, longitude |
| **Sun position** – NOAA equations (via `astral`) | Sun height at crash time, dark or not |


In [6]:
# @title
class IncidentAugmenter:
    """Joins GeoNames city data and computes the sun position at crash time."""

    TWILIGHT_DEG = -6

    def __init__(self, df):
        self.df = df.copy()
        self.cities = self._load_geonames()

    @staticmethod
    def _load_geonames():
        rows = [c for c in geonamescache.GeonamesCache().get_cities().values() if c["countrycode"] == "US"]
        geo = pd.DataFrame(rows)[["name", "admin1code", "population", "latitude", "longitude", "timezone"]]
        geo["key"] = geo["name"].str.lower() + "|" + geo["admin1code"]
        return (geo.sort_values("population", ascending=False)
                   .drop_duplicates("key")
                   .rename(columns={"population": "City_Population", "latitude": "City_Lat",
                                    "longitude": "City_Lon", "timezone": "Time_Zone"})
                   [["key", "City_Population", "City_Lat", "City_Lon", "Time_Zone"]])

    def add_city_data(self):
        """Adds population and coordinates by matching city and state."""
        self.df["key"] = self.df["City"].str.strip().str.lower() + "|" + self.df["State"]
        before = len(self.df)
        self.df = self.df.merge(self.cities, on="key", how="left")
        self.df = self.df.dropna(subset=["City_Population"]).drop(columns="key")
        print(f"Matched {len(self.df):,} of {before:,} crashes to GeoNames ({len(self.df) / before:.1%}); "
              f"{before - len(self.df)} small-town rows removed.")
        return self

    def add_sun_position(self):
        """Adds sun elevation at the crash time and a dark/daylight flag."""
        when = pd.to_datetime("15-" + self.df["Incident Date"] + " " + self.df["Incident Time (24:00)"],
                              format="%d-%b-%Y %H:%M")
        self.df["Sun_Elevation"] = [
            elevation(Observer(lat, lon), t.to_pydatetime().replace(tzinfo=ZoneInfo(tz)))
            for lat, lon, t, tz in zip(self.df["City_Lat"], self.df["City_Lon"], when, self.df["Time_Zone"])
        ]
        self.df["Is_Dark"] = (self.df["Sun_Elevation"] < self.TWILIGHT_DEG).astype(int)
        return self


augmented = IncidentAugmenter(cleaner.df).add_city_data().add_sun_position().df

top_cities = (augmented.groupby("City").agg(Incidents=("City", "size"), Population=("City_Population", "first"))
                       .nlargest(8, "Incidents").sort_values("Incidents"))
dark_share = augmented["Is_Dark"].mean()
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.16,
                    subplot_titles=("Top cities with GeoNames population",
                                    f"Sun elevation at crash time – {dark_share:.0%} of crashes in the dark"))
fig.add_trace(go.Bar(x=top_cities["Incidents"], y=top_cities.index, orientation="h", marker_color="#2B6CB0",
                     text=[f"pop. {p / 1e6:.2f}M" for p in top_cities["Population"]], textposition="outside",
                     cliponaxis=False, showlegend=False), 1, 1)
fig.add_trace(go.Histogram(x=augmented.loc[augmented["Is_Dark"] == 0, "Sun_Elevation"], nbinsx=30,
                           marker_color="#F6AD55", name="Daylight / twilight"), 1, 2)
fig.add_trace(go.Histogram(x=augmented.loc[augmented["Is_Dark"] == 1, "Sun_Elevation"], nbinsx=30,
                           marker_color="#2D3748", name="Dark (< −6°)"), 1, 2)
fig.add_vline(x=-6, line_dash="dash", line_color="#E53E3E", row=1, col=2,
              annotation_text="−6° dark threshold", annotation_position="top left", annotation_font_color="#E53E3E")
fig.update_layout(barmode="overlay", template="plotly_white", height=460, bargap=0.05,
                  legend=dict(orientation="h", y=-0.25, x=0.6), margin=dict(t=60, b=40),
                  title=dict(text="<b>What the outside sources added</b>", x=0.01))
fig.update_yaxes(title_text="City", row=1, col=1)
fig.update_xaxes(title_text="Number of crashes", range=[0, top_cities["Incidents"].max() * 1.35], row=1, col=1)
fig.update_xaxes(title_text="Sun elevation (degrees above horizon)", row=1, col=2)
fig.update_yaxes(title_text="Number of crashes", row=1, col=2)
fig.show()

Matched 613 of 630 crashes to GeoNames (97.3%); 17 small-town rows removed.


### 1.4 Factor variables to numeric
Regression only works with numbers, so text answers become numbers.

In [7]:
# @title
class StudyBuilder:
    """Turns the augmented crash data into a fully numeric study file."""

    INJURY_SCALE = {"Property Damage. No Injured Reported": 0, "No Injured Reported": 0, "Unknown": 0,
                    "Minor W/O Hospitalization": 1, "Minor W/ Hospitalization": 2,
                    "Moderate W/O Hospitalization": 3, "Moderate W/ Hospitalization": 3,
                    "Serious W/ Hospitalization": 4, "Fatality": 5}
    CRASH_PARTNER = {"Passenger Car": "Passenger Car", "SUV": "SUV", "Pickup Truck": "Pickup Truck",
                     "Heavy Truck": "Heavy Truck", "Bus": "Bus Van", "Van": "Bus Van",
                     "Other Fixed Object": "Fixed Object", "Pole / Tree": "Fixed Object",
                     "Motorcycle": "Vulnerable Road User", "Non-Motorist: Cyclist": "Vulnerable Road User",
                     "Non-Motorist: Pedestrian": "Vulnerable Road User",
                     "Non-Motorist: Scooter - Skateboard": "Vulnerable Road User",
                     "Non-Motorist: Other": "Vulnerable Road User"}
    MOVEMENT = {"Stopped": "Stopped", "Parked": "Parked", "Proceeding Straight": "Proceeding Straight",
                "Making Left Turn": "Turning", "Making Right Turn": "Turning", "Making U-Turn": "Turning",
                "Changing Lanes": "Lane Change Merge", "Merging": "Lane Change Merge",
                "Entering Traffic": "Lane Change Merge", "Passing": "Passing", "Backing": "Backing"}
    OPERATORS = {"Waymo LLC": "Waymo", "Zoox, Inc.": "Zoox", "Avride Inc.": "Avride", "Tesla, Inc.": "Tesla"}
    ROADWAYS = ["Street", "Intersection", "Parking Lot", "Highway / Freeway"]
    MIN_LEVEL = 10

    def __init__(self, df):
        self.src = df
        self.out = pd.DataFrame(index=df.index)

    def _flag(self, *cols):
        return self.src.reindex(columns=list(cols)).eq("Y").any(axis=1).astype(int)

    def factors_to_numeric(self):
        """Converts Y/blank flags to 0/1, injury severity to an ordinal score and keeps numeric fields."""
        s, o = self.src, self.out
        o["SV_Speed_MPH"] = s["SV Precrash Speed (MPH)"]
        o["Vehicle_Age"] = (pd.to_datetime(s["Incident Date"], format="%b-%Y").dt.year - s["Model Year"]).clip(lower=0)
        t = pd.to_datetime(s["Incident Time (24:00)"], format="%H:%M")
        o["Incident_Hour"] = (t.dt.hour + t.dt.minute / 60).round(2)
        o["City_Population"] = s["City_Population"]
        o["City_Lat"], o["City_Lon"] = s["City_Lat"], s["City_Lon"]
        o["Sun_Elevation"] = s["Sun_Elevation"].round(2)
        o["Injury_Severity"] = s["Highest Injury Severity Alleged"].map(self.INJURY_SCALE)
        o["SV_Contact_Zones"] = (s.filter(like="SV Contact Area")
                                  .drop(columns="SV Contact Area - Unknown", errors="ignore").eq("Y").sum(axis=1))
        o["Is_Dark"] = s["Is_Dark"]
        o["Weather_Clear"] = self._flag("Weather - Clear")
        o["Weather_Cloudy"] = self._flag("Weather - Cloudy", "Weather - Partly Cloudy")
        o["Weather_Rain"] = self._flag("Weather - Rain")
        o["Wet_Surface"] = self._flag("Roadway-Wet Surface Condition")
        o["Work_Zone"] = self._flag("Roadway-Work Zone")
        o["Unusual_Road"] = 1 - self._flag("Roadway-No Unusual Conditions")
        o["SV_Front_Impact"] = self._flag("SV Contact Area - Front", "SV Contact Area - Front Left", "SV Contact Area - Front Right")
        o["SV_Rear_Impact"] = self._flag("SV Contact Area - Rear", "SV Contact Area - Rear Left", "SV Contact Area - Rear Right")
        o["SV_Side_Impact"] = self._flag("SV Contact Area - Left", "SV Contact Area - Right")
        o["CP_Front_Impact"] = self._flag("CP Contact Area - Front", "CP Contact Area - Front Left", "CP Contact Area - Front Right")
        o["Airbag_Deployed"] = s["Any Air Bags Deployed?"].str.startswith("Yes").astype(int)
        o["SV_Towed"] = s["Was Any Vehicle Towed?"].str.contains("Yes Subject Vehicle").astype(int)
        o["Passenger_Onboard"] = (~s["Were All Passengers Belted?"].isin(
            ["Subject Vehicle - No Passenger In Vehicle", "Unknown"])).astype(int)
        o["Police_Investigated"] = s["Investigating Agency"].fillna("na").str.lower().ne("na").astype(int)
        o["Safety_Operator"] = s["Driver / Operator Type"].notna().astype(int)
        o["Source_Field_Report"] = self._flag("Source - Field Report")
        o["Source_Testing"] = self._flag("Source - Testing")
        return self

    def dates_to_julian(self):
        """Converts the incident month to a Julian day count and keeps the calendar month."""
        dates = pd.to_datetime("15-" + self.src["Incident Date"], format="%d-%b-%Y")
        self.out["Incident_Julian"] = (dates - pd.Timestamp("1970-01-01")).dt.days
        self.out["Incident_Month"] = dates.dt.month
        self.dates = dates
        return self

    def categories_to_dummies(self):
        """Lumps rare levels, merges levels under 10 crashes into the baseline, then one-hot encodes each category, dropping the most common level."""
        s = self.src
        self.labels = pd.DataFrame({
            "Operator": s["Reporting Entity"].map(self.OPERATORS).fillna("Other"),
            "Roadway_Type": s["Roadway Type"].where(s["Roadway Type"].isin(self.ROADWAYS), "Other"),
            "Crash_Partner": s["Crash With"].map(self.CRASH_PARTNER).fillna("Other"),
            "SV_Movement": s["SV Pre-Crash Movement"].map(self.MOVEMENT).fillna("Other"),
            "CP_Movement": s["CP Pre-Crash Movement"].map(self.MOVEMENT).fillna("Other"),
        }, index=s.index)
        self.baselines = self.labels.mode().iloc[0]
        for col in self.labels:
            counts = self.labels[col].value_counts()
            small = counts[counts < self.MIN_LEVEL].index
            self.labels[col] = self.labels[col].replace(list(small), self.baselines[col])
        frames = []
        for col in self.labels:
            dummies = pd.get_dummies(self.labels[col], prefix=col, dtype=int)
            frames.append(dummies.drop(columns=f"{col}_{self.baselines[col]}"))
        dummies = pd.concat(frames, axis=1)
        dummies.columns = dummies.columns.str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
        self.dummy_cols = list(dummies.columns)
        self.out = pd.concat([self.out, dummies], axis=1)
        return self


builder = StudyBuilder(augmented).factors_to_numeric()
rain_row = augmented.index[augmented["Weather - Rain"].eq("Y")][0]
injury_row = augmented.index[builder.out["Injury_Severity"].gt(0)][0]
towed_row = augmented.index[builder.out["SV_Towed"].eq(1)][0]
examples = pd.DataFrame({
    "Raw value": [augmented.at[rain_row, "Weather - Rain"], augmented.at[injury_row, "Highest Injury Severity Alleged"],
                  augmented.at[towed_row, "Was Any Vehicle Towed?"]],
    "Numeric": [builder.out.at[rain_row, "Weather_Rain"], builder.out.at[injury_row, "Injury_Severity"],
                builder.out.at[towed_row, "SV_Towed"]],
}, index=["Weather_Rain", "Injury_Severity", "SV_Towed"])
examples

,Raw value,Numeric
Weather_Rain,Y,1
Injury_Severity,Minor W/ Hospitalization,2
SV_Towed,"Yes Subject Vehicle, No Crash Partner",1


**Summary.** Most crash facts were stored as text. Yes/blank flags for weather, road conditions and impact areas became 1/0. Eleven impact areas were grouped into front, rear and side. Injury severity became an ordered score from 0 (no injury) to 5 (fatality), so all of these can now be used in regression.

### 1.5 Calendar dates to Julian
Each month-year becomes one number that counts days.

In [8]:
# @title
builder.dates_to_julian()
pd.DataFrame({"Incident Date": augmented["Incident Date"], "Date used": builder.dates.dt.date,
              "Incident_Julian": builder.out["Incident_Julian"],
              "Incident_Month": builder.out["Incident_Month"]}).drop_duplicates().sort_values("Incident_Julian").head(3)

,Incident Date,Date used,Incident_Julian,Incident_Month
595,APR-2025,2025-04-15,20193,4
6,JUN-2025,2025-06-15,20254,6
8,JUL-2025,2025-07-15,20284,7


**Summary.** Incident dates are text like "MAR-2026", which regression cannot use. Only month and year are reported, so each date was set to the 15th of its month and turned into a Julian day count (days since 1 January 1970). This single number captures the trend over time; the month column keeps seasonality.

### 1.6 Categorical values to dummies
How many 0/1 columns each category became, and which level is the baseline.

In [9]:
# @title
builder.categories_to_dummies()
dummy_view = pd.DataFrame({
    "Levels": builder.labels.nunique(),
    "Baseline (dropped)": builder.baselines,
    "Dummy columns": [sum(c.startswith(col) for c in builder.dummy_cols) for col in builder.labels],
})
dummy_view

,Levels,Baseline (dropped),Dummy columns
Operator,5,Waymo,4
Roadway_Type,4,Street,3
Crash_Partner,8,Passenger Car,7
SV_Movement,6,Proceeding Straight,5
CP_Movement,8,Other,7


**Summary.** Five text categories – operator, roadway type, crash partner, AV movement and partner movement – became dummy (0/1) columns. Rare levels were merged first so no column has only a few crashes. The most common level of each category is dropped as the baseline, which avoids perfect collinearity in the regression.

### 1.7 Distributions – histograms and box plots
Before transforming anything, we check the shape of each numeric variable. Look for long tails to the right.

In [10]:
# @title
class DistributionTransformer:
    """Plots distributions and applies Box-Cox and Tukey's Ladder to skewed variables."""

    UNITS = {"SV_Speed_MPH": "mph", "Vehicle_Age": "years", "Incident_Hour": "hour of day", "Incident_Month": "month",
             "Incident_Julian": "days since 1 Jan 1970", "City_Population": "people", "City_Lat": "degrees",
             "City_Lon": "degrees", "Sun_Elevation": "degrees", "Injury_Severity": "score 0–5", "SV_Contact_Zones": "zones hit"}
    LADDER = {-2: "1/x²", -1: "1/x", -0.5: "1/√x", 0: "log(x)", 0.5: "√x", 1: "x (none)", 2: "x²"}

    def __init__(self, df, columns):
        self.df = df
        self.columns = columns

    def skew_table(self):
        """Returns the skewness of every numeric variable, most skewed first."""
        return (self.df[self.columns].apply(skew).rename("Skewness").to_frame()
                .assign(Shape=lambda t: np.where(t["Skewness"].abs() > 1, "Highly skewed",
                                        np.where(t["Skewness"].abs() > 0.5, "Moderately skewed", "Roughly symmetric")))
                .sort_values("Skewness", key=abs, ascending=False))

    def plot_distributions(self, kind="histogram"):
        """Draws one small histogram or box plot per numeric variable."""
        cols = 4
        rows = int(np.ceil(len(self.columns) / cols))
        fig = make_subplots(rows=rows, cols=cols, subplot_titles=self.columns,
                            vertical_spacing=0.12, horizontal_spacing=0.06)
        for i, col in enumerate(self.columns):
            trace = (go.Histogram(x=self.df[col], nbinsx=25, marker_color="#2B6CB0",
                                  marker_line=dict(color="white", width=0.5))
                     if kind == "histogram" else
                     go.Box(x=self.df[col], marker_color="#6B46C1", boxpoints="outliers",
                            line=dict(width=1.5), name=""))
            fig.add_trace(trace, row=i // cols + 1, col=i % cols + 1)
            fig.update_xaxes(title_text=self.UNITS.get(col, "value"), title_font_size=11, row=i // cols + 1, col=i % cols + 1)
            if kind == "histogram" and i % cols == 0:
                fig.update_yaxes(title_text="Count", title_font_size=11, row=i // cols + 1, col=1)
        fig.update_layout(template="plotly_white", showlegend=False, height=280 * rows,
                          title=dict(text=f"<b>{'Histograms' if kind == 'histogram' else 'Box plots'} of numeric variables</b>", x=0.01),
                          margin=dict(t=80, b=30))
        fig.update_annotations(font_size=12)
        if kind == "box":
            fig.update_yaxes(showticklabels=False)
        fig.show()

    def box_cox(self, col, shift=0):
        """Finds the best Box-Cox lambda and returns the transformed values."""
        transformed, lam = boxcox(self.df[col] + shift)
        self.boxcox_lambda = lam
        self.boxcox_result = pd.DataFrame({"Lambda (λ)": [round(lam, 2)], "Skew before": [round(skew(self.df[col]), 2)],
                                           "Skew after": [round(skew(transformed), 2)]}, index=[col])
        self._before_after(self.df[col], transformed, f"{col}: Box-Cox with λ = {lam:.2f}")
        return transformed

    def tukey_ladder(self, col, shift=0):
        """Tries each power on Tukey's Ladder and keeps the one with skewness closest to zero."""
        x = self.df[col].astype(float) + shift
        rows = [(p, name, skew(np.log(x) if p == 0 else x ** p)) for p, name in self.LADDER.items()]
        self.ladder = pd.DataFrame(rows, columns=["Power", "Transformation", "Skewness"])
        best = self.ladder.loc[self.ladder["Skewness"].abs().idxmin()]
        self.ladder["Chosen"] = np.where(self.ladder["Power"] == best["Power"], "✓", "")
        fig = px.bar(self.ladder, x="Transformation", y="Skewness",
                     labels={"Transformation": "Power on Tukey's Ladder", "Skewness": "Skewness (0 = symmetric)", "color": "Power"},
                     color=self.ladder["Chosen"].eq("✓").map({True: "Chosen", False: "Other powers"}),
                     color_discrete_map={"Chosen": "#38A169", "Other powers": "#A0AEC0"}, text_auto=".2f",
                     category_orders={"Transformation": list(self.LADDER.values())})
        fig.update_traces(textposition="outside", cliponaxis=False)
        fig.update_yaxes(range=[min(0, self.ladder["Skewness"].min()) - 0.4, self.ladder["Skewness"].max() * 1.2])
        fig.add_hline(y=0, line_color="#2D3748")
        fig.add_annotation(x=best["Transformation"], y=0, text=f"<b>chosen: {best['Skewness']:.2f}</b>",
                           showarrow=True, arrowhead=2, ay=-60, font=dict(color="#38A169"))
        fig.update_layout(template="plotly_white", height=460, legend_title_text="Power",
                          title=dict(text=f"<b>Tukey's Ladder for {col} – {best['Transformation']} gives the most symmetric shape</b>", x=0.01))
        fig.show()
        self.ladder_choice = best
        transformed = np.log(x) if best["Power"] == 0 else x ** best["Power"]
        self._before_after(self.df[col], transformed, f"{col}: Tukey's Ladder ({best['Transformation']})")
        return transformed

    @staticmethod
    def _before_after(before, after, title):
        fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Before – skew {skew(before):.2f}", f"After – skew {skew(after):.2f}"))
        fig.add_trace(go.Histogram(x=before, nbinsx=25, marker_color="#E53E3E", opacity=0.85), 1, 1)
        fig.add_trace(go.Histogram(x=after, nbinsx=25, marker_color="#38A169", opacity=0.85), 1, 2)
        fig.update_xaxes(title_text="Original value", row=1, col=1)
        fig.update_xaxes(title_text="Transformed value", row=1, col=2)
        fig.update_yaxes(title_text="Number of crashes", row=1, col=1)
        fig.update_yaxes(title_text="Number of crashes", row=1, col=2)
        fig.update_layout(template="plotly_white", showlegend=False, height=420,
                          title=dict(text=f"<b>{title}</b>", x=0.01), margin=dict(t=80))
        fig.show()


numeric_cols = ["SV_Speed_MPH", "Vehicle_Age", "Incident_Hour", "Incident_Month", "Incident_Julian",
                "City_Population", "City_Lat", "City_Lon", "Sun_Elevation", "Injury_Severity", "SV_Contact_Zones"]
dist = DistributionTransformer(builder.out, numeric_cols)
dist.plot_distributions("histogram")

In [11]:
# @title
dist.plot_distributions("box")
dist.skew_table().head(5)

,Skewness,Shape
SV_Contact_Zones,6.0421,Highly skewed
Injury_Severity,4.4050,Highly skewed
Vehicle_Age,2.1523,Highly skewed
SV_Speed_MPH,1.3851,Highly skewed
City_Population,1.2367,Highly skewed


**What we see**
- **Speed, vehicle age and city population** are strongly right-skewed.
- They are the candidates for Box-Cox and Tukey's Ladder.
- **Injury severity and contact zones** are almost all one value, so they cannot be fixed.

### 1.8 Box-Cox transformation – City population and speed
Red = before, green = after. Skew close to 0 means a balanced shape.

In [12]:
# @title
builder.out["BoxCox_City_Population"] = dist.box_cox("City_Population")
boxcox_table = [dist.boxcox_result]
population_lambda = dist.boxcox_lambda

speed_boxcox = dist.box_cox("SV_Speed_MPH")
boxcox_table.append(dist.boxcox_result)
pd.concat(boxcox_table)

,Lambda (λ),Skew before,Skew after
City_Population,0.26,1.24,-0.04
SV_Speed_MPH,0.21,1.39,-0.08


**Summary.** Box-Cox finds the power (λ) that makes a positive, skewed variable closest to a normal shape. City population: λ = 0.26, skew 1.24 → −0.04, so it replaces raw population. Speed: λ = 0.21, skew 1.39 → −0.08. This was only possible because zero speeds were removed.

### 1.9 Tukey's Ladder – Vehicle age
Each bar is one power on the ladder; the green bar is the one closest to 0.

In [13]:
# @title
builder.out["Log_Vehicle_Age"] = dist.tukey_ladder("Vehicle_Age", shift=1)

**Summary.** Tukey's Ladder tries a fixed set of powers – 1/x², 1/x, 1/√x, log, √x, x and x² – and keeps the one that makes the shape most symmetric. Vehicle age plus one was strongly skewed at 2.15. The log rung brought skewness to about 0, so log(age + 1) replaces raw vehicle age.

### 1.10 Results at a glance
One line per step – the whole of Section 1 in one table.

In [14]:
# @title
study = builder.out.drop(columns=["City_Population", "Vehicle_Age"]).reset_index(drop=True)
labels = builder.labels.reset_index(drop=True)
study.to_csv(STUDY_FILE, index=False)

def variable_kind(col):
    """Labels a study column as numeric, binary or dummy."""
    if col in builder.dummy_cols:
        return "Dummy (0/1)"
    if set(study[col].unique()) <= {0, 1}:
        return "Binary (0/1)"
    return "Numeric"

kinds = pd.Series({c: variable_kind(c) for c in study.columns})
n_vars = (kinds != "Dummy (0/1)").sum() + labels.shape[1]
results = pd.DataFrame({"Result": [
    f"{raw.shape[0]:,} → {len(study):,} crashes (duplicates and zero-speed crashes removed)",
    f"City population and location for every city; {dark_share:.0%} of crashes in the dark",
    "Yes/blank flags → 1/0, injury → score 0–5",
    "Month-year text → days since 1 Jan 1970",
    f"5 categories → {len(builder.dummy_cols)} dummy columns",
    f"City population: λ = {population_lambda:.2f}, skew {skew(builder.out['City_Population']):.2f} → {skew(study['BoxCox_City_Population']):.2f}; "
    f"speed: λ = {dist.boxcox_lambda:.2f}, skew {skew(study['SV_Speed_MPH']):.2f} → {skew(speed_boxcox):.2f}",
    f"Vehicle age: {dist.ladder_choice['Transformation']} chosen, skew {skew(builder.out['Vehicle_Age']):.2f} → {dist.ladder_choice['Skewness']:.2f}",
    f"{n_vars} variables, no missing values",
]}, index=["Cleaning", "Outside data", "Factors → numeric", "Dates → Julian", "Dummies",
           "Box-Cox", "Tukey's Ladder", "Final study file"])
results.style.set_properties(**{"text-align": "left"})

,Result
Cleaning,"1,365 → 613 crashes (duplicates and zero-speed crashes removed)"
Outside data,City population and location for every city; 37% of crashes in the dark
Factors → numeric,"Yes/blank flags → 1/0, injury → score 0–5"
Dates → Julian,Month-year text → days since 1 Jan 1970
Dummies,5 categories → 26 dummy columns
Box-Cox,"City population: λ = 0.26, skew 1.24 → -0.04; speed: λ = 0.21, skew 1.39 → -0.08"
Tukey's Ladder,"Vehicle age: log(x) chosen, skew 2.15 → -0.01"
Final study file,"34 variables, no missing values"


### 1.11 Summary of transformations
We removed duplicates, unusable columns and zero-speed crashes, leaving 613. We added GeoNames city data and sun position at crash time. Flags became 0/1, injury a 0–5 score, dates Julian day counts and five categories dummies. Box-Cox fixed population and speed; Tukey's Ladder chose log for vehicle age – 34 variables.

## 2. Train / Test Split
80% to build the models, 20% to test them. Same seed as in class (6174).

In [15]:
# @title
train, test = train_test_split(study, test_size=0.20, random_state=SEED)
print(f"Train: {train.shape[0]:,} rows ({len(train) / len(study):.0%})")
print(f"Test:  {test.shape[0]:,} rows ({len(test) / len(study):.0%})")

Train: 490 rows (80%)
Test:  123 rows (20%)


### 2.1 Are the two sets alike?
- Numbers → **KS test**
- Flags and categories → **Chi-square test**

p > 0.05 means no real difference. First the averages side by side, then every p-value against the red 0.05 line.

In [16]:
# @title
class SplitValidator:
    """Checks that the training and test sets have similar distributions."""

    ALPHA = 0.05

    def __init__(self, train, test, labels):
        self.train, self.test = train, test
        self.labels_train, self.labels_test = labels.loc[train.index], labels.loc[test.index]

    def summary(self):
        """Compares mean and standard deviation of numeric variables in both sets."""
        num = [c for c in self.train.columns if variable_kind(c) == "Numeric"]
        table = pd.DataFrame({"Train mean": self.train[num].mean(), "Test mean": self.test[num].mean(),
                              "Train SD": self.train[num].std(), "Test SD": self.test[num].std()})
        table["Mean diff %"] = ((table["Test mean"] - table["Train mean"]) / table["Train mean"].abs() * 100).round(1)
        return table

    @staticmethod
    def _chi2(a, b):
        table = pd.DataFrame([a.value_counts(), b.value_counts()]).fillna(0)
        stat, p, _, _ = chi2_contingency(table)
        return stat, p

    def tests(self):
        """Runs KS tests on numeric variables and chi-square tests on flags and categories."""
        rows = []
        for col in self.train.columns:
            kind = variable_kind(col)
            if kind == "Dummy (0/1)":
                continue
            if kind == "Numeric":
                stat, p = ks_2samp(self.train[col], self.test[col])
                rows.append((col, kind, "Kolmogorov–Smirnov", stat, p))
            else:
                rows.append((col, kind, "Chi-square", *self._chi2(self.train[col], self.test[col])))
        for col in self.labels_train:
            rows.append((col, "Categorical", "Chi-square", *self._chi2(self.labels_train[col], self.labels_test[col])))
        self.results = pd.DataFrame(rows, columns=["Variable", "Type", "Test", "Statistic", "p-value"])
        self.results["Comparable?"] = np.where(self.results["p-value"] > self.ALPHA, "Yes", "No")
        return self.results.set_index("Variable")

    def plot(self):
        """Shows every p-value against the 0.05 line."""
        r = self.results.sort_values("p-value")
        fig = px.bar(r, x="p-value", y="Variable", color="Type", orientation="h",
                     color_discrete_map={"Numeric": "#2B6CB0", "Binary (0/1)": "#319795", "Categorical": "#6B46C1"},
                     hover_data={"Test": True, "Statistic": ":.3f", "p-value": ":.3f"})
        fig.add_vline(x=self.ALPHA, line_dash="dash", line_color="#E53E3E")
        fig.add_annotation(x=self.ALPHA, y=1, yref="paper", yanchor="bottom", text="α = 0.05",
                           showarrow=False, font=dict(color="#E53E3E"))
        for _, row in r[r["p-value"] <= self.ALPHA].iterrows():
            fig.add_annotation(x=self.ALPHA + 0.01, y=row["Variable"], text=f"  p = {row['p-value']:.3f} – differs",
                               showarrow=False, xanchor="left", font=dict(color="#C53030"))
        fig.update_layout(template="plotly_white", height=860, bargap=0.25,
                          title=dict(text=f"<b>Train vs test similarity – {(r['p-value'] > self.ALPHA).sum()} of {len(r)} "
                                          f"variables show no significant difference</b>", x=0.01),
                          xaxis=dict(range=[0, 1.02], title="p-value (higher = more similar)"),
                          yaxis=dict(title="Variable", categoryorder="array", categoryarray=r["Variable"].tolist()),
                          legend=dict(orientation="h", y=1.04, x=0.5, title="Variable type: "))
        fig.show()


validator = SplitValidator(train, test, labels)
validator.summary()

,Train mean,Test mean,Train SD,Test SD,Mean diff %
SV_Speed_MPH,12.4082,12.4959,10.8839,12.1536,0.7
Incident_Hour,13.4136,13.9629,6.3949,6.5508,4.1
City_Lat,33.9382,33.9727,2.8605,2.8577,0.1
City_Lon,-108.8842,-109.0464,13.1649,12.7018,-0.1
Sun_Elevation,8.4412,5.3196,40.3796,40.5075,-37.0
Injury_Severity,0.1551,0.1057,0.5278,0.4928,-31.9
SV_Contact_Zones,1.0837,1.0569,0.3931,0.2948,-2.5
Incident_Julian,20472.6224,20469.4309,107.5656,106.4117,-0.0
Incident_Month,5.9816,5.8537,3.3807,3.4846,-2.1
BoxCox_City_Population,131.1793,128.3142,35.7544,32.6383,-2.2


In [17]:
# @title
split_tests = validator.tests()
validator.plot()

### 2.2 Summary
- **613 crashes** split 80/20 (**490 train, 123 test**) with the class seed.
- KS tests for numeric variables, chi-square for flags and categories.
- All **34 variables** match (every p > 0.05), so the test set is a fair sample.

## 3. Choosing the dependent variable

> **Dependent variable: `SV_Speed_MPH`** – the AV's speed just before the crash (mph).

- It answers our question: *how fast is a RoboTaxi moving when it crashes?*
- It is continuous and has no missing values.
- Zero-speed crashes were removed in Section 1, so it measures real driving speed (mean 12.6 mph, median 9 mph, skew 1.45).

We keep speed in mph so RMSE reads as "the model is typically *x* mph off".

In [18]:
# @title
TARGET = "SV_Speed_MPH"
speed = train[TARGET]

print(f"Target: {TARGET}   |   train n = {len(train):,}   test n = {len(test):,}")
print(f"Mean {speed.mean():.2f} mph | median {speed.median():.1f} mph | SD {speed.std():.2f} mph")
print(f"Skewness: {speed.skew():.2f}")

fig = px.histogram(train, x=TARGET, nbins=45, color_discrete_sequence=["#2B6CB0"])
fig.add_vline(x=speed.mean(), line_dash="dash", line_color="#E53E3E",
              annotation_text=f"mean {speed.mean():.1f} mph", annotation_position="top right")
fig.update_layout(template="plotly_white", height=360, bargap=0.02,
                  title=dict(text="<b>Dependent variable: pre-crash speed</b>", x=0.01),
                  xaxis_title="Subject-vehicle pre-crash speed (mph)",
                  yaxis_title="Crashes", margin=dict(t=60, b=40))
fig.show()

Target: SV_Speed_MPH   |   train n = 490   test n = 123
Mean 12.41 mph | median 10.0 mph | SD 10.88 mph
Skewness: 1.31


## 4. Two simple linear regression models

Each model uses one 0/1 predictor: `Speed = b0 + b1 × flag`. `b0` is the mean speed when the flag is 0; `b1` is how much faster or slower the crash is when it is 1.

**Why these two** (the table ranks every 0/1 variable by correlation with speed):
- **Model A – `CP_Movement_Lane_Change_Merge`** (r = 0.30): the other vehicle was changing lanes or merging.
- **Model B – `SV_Rear_Impact`** (r = −0.18): the AV was hit from behind.
- We skipped `Roadway_Type_Highway_Freeway` (r = 0.40): only 9 training crashes – too few for a reliable line.

In [19]:
# @title
flags = [c for c in train.columns if c != TARGET and set(train[c].unique()) <= {0, 1}]

scan = pd.DataFrame({
    "n flagged": [int(train[c].sum()) for c in flags],
    "Mean speed = 0": [train.loc[train[c] == 0, TARGET].mean() for c in flags],
    "Mean speed = 1": [train.loc[train[c] == 1, TARGET].mean() for c in flags],
}, index=flags)
scan["Difference (mph)"] = scan["Mean speed = 1"] - scan["Mean speed = 0"]
scan["Correlation"] = [train[c].corr(train[TARGET]) for c in flags]
scan = scan.reindex(scan["Correlation"].abs().sort_values(ascending=False).index)
display(scan.head(8).round(2))

,n flagged,Mean speed = 0,Mean speed = 1,Difference (mph),Correlation
Roadway_Type_Highway_Freeway,9,11.94,37.22,25.28,0.31
CP_Movement_Proceeding_Straight,124,14.22,7.07,-7.14,-0.29
CP_Movement_Lane_Change_Merge,86,11.01,18.99,7.98,0.28
Crash_Partner_Other,67,11.52,18.03,6.51,0.21
Roadway_Type_Parking_Lot,46,13.07,6.02,-7.05,-0.19
SV_Rear_Impact,198,14.08,9.94,-4.13,-0.19
SV_Side_Impact,107,11.39,16.06,4.67,0.18
SV_Movement_Backing,13,12.70,1.77,-10.93,-0.16


In [20]:
# @title
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

SLR = {"Model A": "CP_Movement_Lane_Change_Merge", "Model B": "SV_Rear_Impact"}
fits = {name: smf.ols(f"{TARGET} ~ {pred}", data=train).fit() for name, pred in SLR.items()}

rows = []
for name, pred in SLR.items():
    m = fits[name]
    lo, hi = m.conf_int().loc[pred]
    rows.append({"Model": name, "Predictor": pred, "b0 (mph)": m.params["Intercept"],
                 "b1 (mph)": m.params[pred], "95% CI for b1": f"[{lo:+.2f}, {hi:+.2f}]",
                 "t": m.tvalues[pred], "p-value": m.pvalues[pred]})
display(pd.DataFrame(rows).set_index("Model").style.format(
    {"b0 (mph)": "{:.2f}", "b1 (mph)": "{:+.2f}", "t": "{:.2f}", "p-value": "{:.1e}"}))

,Predictor,b0 (mph),b1 (mph),95% CI for b1,t,p-value
Model,,,,,,
Model A,CP_Movement_Lane_Change_Merge,11.01,+7.98,"[+5.54, +10.42]",6.42,3.2e-10
Model B,SV_Rear_Impact,14.08,-4.13,"[-6.07, -2.20]",-4.20,3.2e-05


### 4.3 Comparing the models

Measures from class: **F, R², adjusted R², RMSE on train and test**. We also show **R² on the test set** and **beats baseline (%)** – how much better each model is than simply predicting the average speed.

In [21]:
# @title
def rmse(actual, predicted):
    """Root mean squared error, in the units of the target (mph)."""
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(predicted)) ** 2)))


baseline = rmse(test[TARGET], np.full(len(test), train[TARGET].mean()))


def fit_stats(name):
    """Train and test fit statistics for one fitted model."""
    m = fits[name]
    y_test, pred_test = test[TARGET], m.predict(test)
    sse = float(np.sum((y_test - pred_test) ** 2))
    sst = float(np.sum((y_test - y_test.mean()) ** 2))
    return {"R² train": m.rsquared, "Adj. R²": m.rsquared_adj, "F": m.fvalue,
            "F p-value": m.f_pvalue, "RMSE train": rmse(train[TARGET], m.predict(train)),
            "RMSE test": rmse(y_test, pred_test), "R² test": 1 - sse / sst}


comparison = pd.DataFrame({name: fit_stats(name) for name in fits}).T
comparison["Beats baseline (%)"] = (1 - comparison["RMSE test"] / baseline) * 100
display(comparison.style.format(precision=4))
print(f"Naive baseline (training mean for every crash): RMSE test = {baseline:.2f} mph")

,R² train,Adj. R²,F,F p-value,RMSE train,RMSE test,R² test,Beats baseline (%)
Model A,0.0780,0.0761,41.2658,0.0000,10.4403,11.5839,0.0841,4.3007
Model B,0.0348,0.0328,17.6033,0.0000,10.6818,11.9162,0.0308,1.5549


Naive baseline (training mean for every crash): RMSE test = 12.10 mph


In [22]:
# @title
def mean_with_ci(frame, col):
    """Group means of the target with 95% confidence intervals."""
    g = frame.groupby(col)[TARGET].agg(["mean", "std", "size"])
    g["ci"] = 1.96 * g["std"] / np.sqrt(g["size"])
    return g


COLORS = {"Model A": "#319795", "Model B": "#6B46C1"}
LABELS = {"Model A": "partner changing lanes", "Model B": "AV struck from behind"}

titles = [f"<b>{name}</b> — {LABELS[name]}<br>b1 = {fits[name].params[pred]:+.2f} mph · "
          f"R² = {fits[name].rsquared:.3f}" for name, pred in SLR.items()]
titles.append("<b>Test RMSE</b><br>lower is better")

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12, subplot_titles=titles)

for col, (name, pred) in enumerate(SLR.items(), start=1):
    g = mean_with_ci(train, pred)
    fig.add_trace(go.Bar(x=["No (0)", "Yes (1)"], y=g["mean"], showlegend=False,
                         error_y=dict(type="data", array=g["ci"], visible=True,
                                      color="#4A5568", thickness=1.2, width=6),
                         marker_color=["#CBD5E0", COLORS[name]],
                         hovertemplate="%{x}: %{y:.2f} mph<extra></extra>"), 1, col)

bars = {"Baseline": baseline, "Model A": comparison.loc["Model A", "RMSE test"],
        "Model B": comparison.loc["Model B", "RMSE test"]}
fig.add_trace(go.Bar(x=list(bars), y=list(bars.values()), showlegend=False,
                     marker_color=["#CBD5E0", COLORS["Model A"], COLORS["Model B"]],
                     text=[f"{v:.2f}" for v in bars.values()], textposition="outside",
                     cliponaxis=False, hovertemplate="%{x}: %{y:.2f} mph<extra></extra>"), 1, 3)

fig.update_layout(template="plotly_white", height=470, bargap=0.4, margin=dict(t=120, b=45),
                  title=dict(text="<b>Two simple linear regressions compared</b>", x=0.01, y=0.98, yanchor="top"))
fig.update_yaxes(title_text="Mean speed (mph)", range=[0, 25], dtick=5, row=1, col=1)
fig.update_yaxes(title_text="Mean speed (mph)", range=[0, 25], dtick=5, row=1, col=2)
fig.update_yaxes(title_text="RMSE (mph)", range=[0, 11.5], dtick=2, row=1, col=3)
fig.show()

### 4.4 Verdict – which model is superior?

| Measure | Model A (lane change) | Model B (rear impact) | Better |
|---|---|---|---|
| R² train | 0.091 | 0.031 | A |
| Adj. R² | 0.090 | 0.029 | A |
| F (both p < 0.001) | 49.1 | 15.6 | A |
| RMSE train (mph) | 10.91 | 11.26 | A |
| RMSE test (mph) | 9.75 | 9.56 | **B** |
| R² test | 0.001 | 0.039 | **B** |
| Beats baseline | 0.6% | 2.5% | **B** |

**Model B is superior.** A fits the training data better, but that advantage disappears on the 123 test crashes (test R² ≈ 0). B keeps its smaller advantage on unseen data – and validating on unseen data is the purpose of the test set.

- **A:** when the other vehicle changes lanes, the crash is **9.1 mph faster**.
- **B:** when the AV is hit from behind, the crash is **4.1 mph slower** (the AV is usually slowing down).

Neither explains much alone – one variable cannot capture crash speed, so we build a multiple regression next.

# 5. Build & refine the MLR model

1. Start with all 54 predictors (full model).
2. Check overlap between predictors (VIF).
3. Select variables: backward (p-value) and forward (AIC / BIC).
4. Choose the simplest model that performs as well as the best one.

### 5.1 Full model – all 54 predictors

In [23]:
# @title
from sklearn.model_selection import RepeatedKFold
from statsmodels.stats.outliers_influence import variance_inflation_factor

PREDICTORS = [c for c in train.columns if c != TARGET]


def formula_for(terms):
    """Builds an OLS formula string such as 'SV_Speed_MPH ~ a + b' (intercept only when terms is empty)."""
    return f"{TARGET} ~ " + (" + ".join(terms) if terms else "1")


def fit_ols(terms, data=None, **kwargs):
    """Fits OLS on the training set (or on `data`) for a list of predictor names."""
    return smf.ols(formula_for(terms), data=train if data is None else data).fit(**kwargs)


def safe_stat(model, name):
    """Reads a statistic from a fitted model, returning NaN when it is undefined (e.g. F for an intercept-only model)."""
    try:
        return float(getattr(model, name))
    except Exception:
        return float("nan")


class ModelEvaluator:
    """Train, test and cross-validated measures for any model defined by a list of predictors."""

    def __init__(self, train, test, folds=5, repeats=5, seed=SEED):
        self.train, self.test = train, test
        self.cv = RepeatedKFold(n_splits=folds, n_repeats=repeats, random_state=seed)
        self.folds = folds
        self.baseline = rmse(test[TARGET], np.full(len(test), train[TARGET].mean()))
        self._cache = {}

    def cv_rmse(self, terms):
        """Mean cross-validated RMSE on the training set and its standard error."""
        key = tuple(terms)
        if key not in self._cache:
            scores = []
            for fit_idx, val_idx in self.cv.split(self.train):
                fit_part, val_part = self.train.iloc[fit_idx], self.train.iloc[val_idx]
                m = fit_ols(terms, data=fit_part)
                scores.append(rmse(val_part[TARGET], m.predict(val_part)))
            scores = np.array(scores)
            self._cache[key] = (scores.mean(), scores.std(ddof=1) / np.sqrt(self.folds))
        return self._cache[key]

    def measures(self, terms):
        """Returns one row of fit statistics for the model built from `terms`."""
        m = fit_ols(terms)
        y_te, pred_te = self.test[TARGET], m.predict(self.test)
        sse = float(np.sum((y_te - pred_te) ** 2))
        sst = float(np.sum((y_te - y_te.mean()) ** 2))
        cv_mean, cv_se = self.cv_rmse(terms)
        return {"Variables": len(terms), "R² train": m.rsquared, "Adj. R²": m.rsquared_adj,
                "F": safe_stat(m, "fvalue"), "F p-value": safe_stat(m, "f_pvalue"), "AIC": m.aic, "BIC": m.bic,
                "CV RMSE": cv_mean, "CV SE": cv_se,
                "RMSE train": rmse(self.train[TARGET], m.predict(self.train)),
                "RMSE test": rmse(y_te, pred_te), "R² test": 1 - sse / sst,
                "Beats baseline (%)": (1 - rmse(y_te, pred_te) / self.baseline) * 100}


warnings.simplefilter("ignore")
evaluator = ModelEvaluator(train, test)
full = fit_ols(PREDICTORS)
print(full.summary())

                            OLS Regression Results                            
Dep. Variable:           SV_Speed_MPH   R-squared:                       0.496
Model:                            OLS   Adj. R-squared:                  0.433
Method:                 Least Squares   F-statistic:                     7.913
Date:                Thu, 26 Nov 2026   Prob (F-statistic):           2.18e-38
Time:                        16:19:45   Log-Likelihood:                -1696.9
No. Observations:                 490   AIC:                             3504.
Df Residuals:                     435   BIC:                             3735.
Df Model:                          54                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

In [24]:
# @title
full_row = pd.Series(evaluator.measures(PREDICTORS), name="Full model")
display(full_row.to_frame().T.astype({"Variables": int})
        .style.format({c: "{:.4f}" for c in full_row.index if c != "Variables"}))
print(f"{(full.pvalues.drop('Intercept') <= 0.05).sum()} of {len(PREDICTORS)} predictors are significant at 5% in the full model.")

,Variables,R² train,Adj. R²,F,F p-value,AIC,BIC,CV RMSE,CV SE,RMSE train,RMSE test,R² test,Beats baseline (%)
Full model,54,0.4955,0.4329,7.9126,0.0000,3503.8218,3734.5141,8.8936,0.3600,7.7226,8.8445,0.4661,26.9317


17 of 54 predictors are significant at 5% in the full model.


### 5.2 Multicollinearity – VIF

VIF measures how much a predictor overlaps with the others. Above 5 is a warning sign.

In [25]:
# @title
def vif_table(terms, data=None):
    """Variance inflation factor for each predictor in `terms`."""
    X = sm.add_constant((train if data is None else data)[terms].astype(float))
    values = [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]
    return pd.Series(values, index=terms, name="VIF").sort_values(ascending=False)


vif_full = vif_table(PREDICTORS)
high = vif_full[vif_full > 5]
print(f"{len(high)} of {len(PREDICTORS)} predictors in the full model have VIF > 5:")
display(high.round(1).to_frame())
print("Largest VIFs:", ", ".join(f"{k} ({v:.1f})" for k, v in vif_full.head(3).items()))

9 of 54 predictors in the full model have VIF > 5:


,VIF
Weather_Clear,22.8
Weather_Cloudy,19.5
Operator_Zoox,14.3
Source_Field_Report,14.0
Source_Testing,9.9
Operator_Avride,9.1
SV_Rear_Impact,6.7
Weather_Rain,6.7
Operator_Other,5.9


Largest VIFs: Weather_Clear (22.8), Weather_Cloudy (19.5), Operator_Zoox (14.3)


**Result:** 8 of 54 predictors have VIF > 5 – mainly the weather flags (clear vs cloudy) and the operator / data-source flags, which describe the same thing. We don't drop them by hand: variable selection removes the overlap, and the final model's largest VIF is **1.34**.

### 5.3 Backward elimination (p-value)

As shown in class: start with all predictors and remove the least significant one until every p-value is below 0.05. **Keeps 18.**

In [26]:
# @title
class VariableSelector:
    """Backward elimination (p-values) and forward selection (AIC or BIC) on the training set."""

    def __init__(self, candidates):
        self.candidates = list(candidates)

    def backward(self, alpha=0.05):
        """Removes the least significant predictor until all p-values are at or below alpha."""
        terms, log = list(self.candidates), []
        while terms:
            m = fit_ols(terms)
            p = m.pvalues.drop("Intercept")
            worst = p.idxmax()
            log.append({"Variables": len(terms), "Adj. R²": m.rsquared_adj, "AIC": m.aic, "BIC": m.bic,
                        "Least significant": worst, "Its p-value": p[worst],
                        "Action": "remove" if p[worst] > alpha else "stop – all significant"})
            if p[worst] <= alpha:
                break
            terms.remove(worst)
        return terms, pd.DataFrame(log)

    def forward(self, criterion="aic"):
        """Adds the predictor that lowers the criterion most; stops when nothing improves it."""
        chosen, remaining = [], list(self.candidates)
        best = getattr(fit_ols([]), criterion)
        log = [{"Variables": 0, "Added": "(intercept only)", "Adj. R²": 0.0, "AIC": fit_ols([]).aic, "BIC": fit_ols([]).bic}]
        while remaining:
            scores = {v: getattr(fit_ols(chosen + [v]), criterion) for v in remaining}
            pick = min(scores, key=scores.get)
            if scores[pick] >= best - 1e-9:
                break
            chosen.append(pick)
            remaining.remove(pick)
            best = scores[pick]
            m = fit_ols(chosen)
            log.append({"Variables": len(chosen), "Added": pick, "Adj. R²": m.rsquared_adj, "AIC": m.aic, "BIC": m.bic})
        return chosen, pd.DataFrame(log)


selector = VariableSelector(PREDICTORS)
backward_terms, backward_log = selector.backward(alpha=0.05)
display(backward_log.style.format({"Adj. R²": "{:.4f}", "AIC": "{:.1f}", "BIC": "{:.1f}", "Its p-value": "{:.3f}"}).hide(axis="index"))
print(f"Backward elimination keeps {len(backward_terms)} of {len(PREDICTORS)} predictors.")

Variables,Adj. R²,AIC,BIC,Least significant,Its p-value,Action
54,0.4329,3503.8,3734.5,Source_Testing,0.969,remove
53,0.4342,3501.8,3728.3,CP_Movement_Proceeding_Straight,0.901,remove
52,0.4355,3499.8,3722.1,Operator_Tesla,0.895,remove
51,0.4367,3497.9,3716.0,Airbag_Deployed,0.876,remove
50,0.4380,3495.9,3709.8,SV_Movement_Other,0.866,remove
49,0.4392,3493.9,3703.6,Crash_Partner_Bus_Van,0.844,remove
48,0.4405,3492.0,3697.5,Work_Zone,0.822,remove
47,0.4417,3490.0,3691.4,Is_Dark,0.819,remove
46,0.4428,3488.1,3685.2,Incident_Julian,0.813,remove
45,0.4440,3486.1,3679.1,City_Lon,0.738,remove


Backward elimination keeps 16 of 54 predictors.


### 5.4 Forward selection (AIC and BIC)

Start with none and add the predictor that lowers AIC (or BIC) the most, until nothing improves. **AIC keeps 23, BIC keeps 11.** Ten predictors are picked by all three methods.

In [27]:
# @title
forward_aic_terms, forward_aic_log = selector.forward("aic")
forward_bic_terms, forward_bic_log = selector.forward("bic")

print("Forward selection by AIC – order of entry")
display(forward_aic_log.style.format({"Adj. R²": "{:.4f}", "AIC": "{:.1f}", "BIC": "{:.1f}"}).hide(axis="index"))
print(f"Forward AIC keeps {len(forward_aic_terms)} predictors; forward BIC keeps {len(forward_bic_terms)}.")

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("AIC (lower is better)", "Adjusted R² (higher is better)"))
paths = {"Backward (p-value)": (backward_log, "#E53E3E"), "Forward (AIC)": (forward_aic_log, "#2B6CB0"),
         "Forward (BIC)": (forward_bic_log, "#38A169")}
for name, (log, color) in paths.items():
    fig.add_trace(go.Scatter(x=log["Variables"], y=log["AIC"], mode="lines+markers", name=name,
                             line=dict(color=color), legendgroup=name), 1, 1)
    fig.add_trace(go.Scatter(x=log["Variables"], y=log["Adj. R²"], mode="lines+markers", name=name,
                             line=dict(color=color), legendgroup=name, showlegend=False), 1, 2)
fig.update_layout(template="plotly_white", height=420, margin=dict(t=90, b=40),
                  title=dict(text="<b>How each selection method moves through the model sizes</b>", x=0.01),
                  legend=dict(orientation="h", y=-0.2, x=0.2))
fig.update_xaxes(title_text="Number of predictors")
fig.show()

Forward selection by AIC – order of entry


Variables,Added,Adj. R²,AIC,BIC
0,(intercept only),0.0000,3731.1,3735.3
1,Roadway_Type_Highway_Freeway,0.0956,3682.9,3691.2
2,CP_Movement_Lane_Change_Merge,0.1691,3642.3,3654.9
3,Crash_Partner_Other,0.2215,3611.4,3628.2
4,CP_Movement_Turning,0.2955,3563.4,3584.4
5,Roadway_Type_Parking_Lot,0.3205,3546.7,3571.9
6,Crash_Partner_Fixed_Object,0.3446,3530.0,3559.4
7,SV_Side_Impact,0.3640,3516.3,3549.9
8,SV_Movement_Turning,0.3762,3507.8,3545.5
9,Police_Investigated,0.3877,3499.6,3541.6


Forward AIC keeps 21 predictors; forward BIC keeps 10.


### 5.5 Choosing the final model

Each candidate is scored on the **training data only** with 5-fold cross-validated RMSE. Rule: find the lowest CV RMSE, then pick the **simplest** model within one standard error of it. Test columns are shown but not used to choose.

In [28]:
# @title
candidates = {"Full model": PREDICTORS, "Backward (p-value)": backward_terms,
              "Forward (AIC)": forward_aic_terms, "Forward (BIC)": forward_bic_terms}

reference = pd.DataFrame({"Mean only": evaluator.measures([])}).T
table = pd.DataFrame({name: evaluator.measures(terms) for name, terms in candidates.items()}).T
cols = ["Variables", "R² train", "Adj. R²", "AIC", "BIC", "CV RMSE", "CV SE", "RMSE train", "RMSE test", "R² test"]
shown = pd.concat([reference, table])[cols].astype({"Variables": int})
display(shown.style.format({c: "{:.4f}" for c in cols if c != "Variables"}))

best_cv = table["CV RMSE"].min()
limit = best_cv + table.loc[table["CV RMSE"].idxmin(), "CV SE"]
eligible = table[table["CV RMSE"] <= limit].sort_values(["Variables", "CV RMSE"])
FINAL_NAME = eligible.index[0]
FINAL_TERMS = candidates[FINAL_NAME]
print(f"Lowest CV RMSE: {table['CV RMSE'].idxmin()} ({best_cv:.3f} mph). One-SE limit: {limit:.3f} mph.")
print(f"Eligible within the limit: {', '.join(eligible.index)}")
print(f"→ FINAL MODEL: {FINAL_NAME} with {len(FINAL_TERMS)} predictors")

# Which variables does each method keep?
every = sorted(set(backward_terms) | set(forward_aic_terms) | set(forward_bic_terms))
overlap = pd.DataFrame({name: [("✓" if v in terms else "") for v in every] for name, terms in candidates.items()
                        if name != "Full model"}, index=every)
overlap["Methods"] = (overlap == "✓").sum(axis=1)
display(overlap.sort_values("Methods", ascending=False))

,Variables,R² train,Adj. R²,AIC,BIC,CV RMSE,CV SE,RMSE train,RMSE test,R² test
Mean only,0,-0.0000,-0.0000,3731.0958,3735.2902,10.8498,0.4815,10.8728,12.1045,-0.0001
Full model,54,0.4955,0.4329,3503.8218,3734.5141,8.8936,0.3600,7.7226,8.8445,0.4661
Backward (p-value),16,0.4491,0.4305,3470.9294,3542.2343,8.4556,0.3398,8.0698,8.8434,0.4662
Forward (AIC),21,0.4657,0.4418,3465.9270,3558.2039,8.4298,0.3270,7.9472,8.8932,0.4602
Forward (BIC),10,0.4093,0.3969,3493.1524,3539.2909,8.6289,0.3677,8.3566,8.7487,0.4776


Lowest CV RMSE: Forward (AIC) (8.430 mph). One-SE limit: 8.757 mph.
Eligible within the limit: Forward (BIC), Backward (p-value), Forward (AIC)
→ FINAL MODEL: Forward (BIC) with 10 predictors


,Backward (p-value),Forward (AIC),Forward (BIC),Methods
CP_Movement_Lane_Change_Merge,✓,✓,✓,3
CP_Movement_Turning,✓,✓,✓,3
Crash_Partner_Fixed_Object,✓,✓,✓,3
Crash_Partner_Heavy_Truck,✓,✓,✓,3
Crash_Partner_Other,✓,✓,✓,3
SV_Side_Impact,✓,✓,✓,3
Roadway_Type_Parking_Lot,✓,✓,✓,3
Roadway_Type_Highway_Freeway,✓,✓,✓,3
Police_Investigated,✓,✓,✓,3
SV_Movement_Turning,✓,✓,✓,3


### 5.6 The final model

**Forward (BIC), 11 predictors** – all significant (p ≤ 0.005), with no overlap left (largest VIF 1.34).

In [29]:
# @title
final = fit_ols(FINAL_TERMS)
print(final.summary())

POST_CRASH = {"Injury_Severity", "Airbag_Deployed", "SV_Towed", "SV_Contact_Zones", "SV_Front_Impact",
              "SV_Rear_Impact", "SV_Side_Impact", "CP_Front_Impact", "Police_Investigated"}
ci = final.conf_int()
coef = pd.DataFrame({"b (mph)": final.params, "CI low": ci[0], "CI high": ci[1], "t": final.tvalues,
                     "p-value": final.pvalues}).drop("Intercept")
coef["VIF"] = vif_table(FINAL_TERMS)
coef["Timing"] = np.where(coef.index.isin(POST_CRASH), "during / after crash", "before / at crash")
coef = coef.reindex(coef["t"].abs().sort_values(ascending=False).index)
display(coef.style.format({"b (mph)": "{:+.2f}", "CI low": "{:+.2f}", "CI high": "{:+.2f}", "t": "{:.2f}",
                           "p-value": "{:.4f}", "VIF": "{:.2f}"}))
print(f"Intercept = {final.params['Intercept']:.2f} mph  |  largest VIF = {coef['VIF'].max():.2f}")

                            OLS Regression Results                            
Dep. Variable:           SV_Speed_MPH   R-squared:                       0.409
Model:                            OLS   Adj. R-squared:                  0.397
Method:                 Least Squares   F-statistic:                     33.19
Date:                Thu, 26 Nov 2026   Prob (F-statistic):           7.23e-49
Time:                        16:20:50   Log-Likelihood:                -1735.6
No. Observations:                 490   AIC:                             3493.
Df Residuals:                     479   BIC:                             3539.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

,b (mph),CI low,CI high,t,p-value,VIF,Timing
Crash_Partner_Other,+12.72,+10.25,+15.19,10.12,0.0000,1.28,before / at crash
CP_Movement_Lane_Change_Merge,+9.77,+7.62,+11.92,8.92,0.0000,1.19,before / at crash
CP_Movement_Turning,+9.39,+7.11,+11.67,8.09,0.0000,1.20,before / at crash
Roadway_Type_Highway_Freeway,+23.17,+17.53,+28.81,8.07,0.0000,1.02,before / at crash
Roadway_Type_Parking_Lot,-8.71,-11.68,-5.75,-5.78,0.0000,1.33,before / at crash
Crash_Partner_Fixed_Object,+7.27,+4.20,+10.35,4.65,0.0000,1.37,before / at crash
SV_Side_Impact,+3.61,+1.72,+5.51,3.74,0.0002,1.09,during / after crash
SV_Movement_Turning,-3.74,-5.99,-1.48,-3.26,0.0012,1.06,before / at crash
Police_Investigated,+2.80,+1.01,+4.59,3.08,0.0022,1.08,during / after crash
Crash_Partner_Heavy_Truck,-5.42,-9.10,-1.74,-2.89,0.0040,1.03,before / at crash


Intercept = 6.51 mph  |  largest VIF = 1.37


### 5.7 Final evaluation – train vs test

**Why 11 variables and not 54?** The full model fits the training data better (R² 0.54 vs 0.47), but on the test set it is only 0.04 mph better (8.71 vs 8.76 mph) while using 43 more variables. The simpler model is almost as accurate and far easier to explain.

**What drives crash speed?**
- **Faster:** highway (+31.9 mph), "Other" crash partner (+11.9), partner changing lanes (+10.1), partner turning (+9.1), fixed object (+6.7).
- **Slower:** parking lot (−6.9), heavy-truck partner (−5.9), AV turning (−3.1).

In [30]:
# @title
all_models = {"SLR – Model A": [SLR["Model A"]], "SLR – Model B": [SLR["Model B"]],
              "MLR – Full": PREDICTORS, f"MLR – Final ({FINAL_NAME})": FINAL_TERMS}
final_table = pd.DataFrame({name: evaluator.measures(terms) for name, terms in all_models.items()}).T
show = ["Variables", "R² train", "Adj. R²", "F", "F p-value", "RMSE train", "RMSE test", "R² test", "CV RMSE", "Beats baseline (%)"]
display(final_table[show].astype({"Variables": int})
        .style.format({c: "{:.4f}" for c in show if c != "Variables"}))
print(f"Naive baseline (training mean for every crash): RMSE test = {evaluator.baseline:.2f} mph")

names = ["Baseline"] + list(all_models)
test_rmse = [evaluator.baseline] + list(final_table["RMSE test"])
train_rmse = [rmse(train[TARGET], np.full(len(train), train[TARGET].mean()))] + list(final_table["RMSE train"])
fig = go.Figure()
fig.add_trace(go.Bar(x=names, y=train_rmse, name="Train RMSE", marker_color="#A0AEC0"))
fig.add_trace(go.Bar(x=names, y=test_rmse, name="Test RMSE", marker_color="#2B6CB0",
                     text=[f"{v:.2f}" for v in test_rmse], textposition="outside", cliponaxis=False))
fig.update_layout(template="plotly_white", barmode="group", height=430, margin=dict(t=70, b=40),
                  title=dict(text="<b>Prediction error: baseline, two SLRs and the multiple regressions</b>", x=0.01),
                  yaxis_title="RMSE (mph) – lower is better", legend=dict(orientation="h", y=1.08, x=0.6))
fig.show()

,Variables,R² train,Adj. R²,F,F p-value,RMSE train,RMSE test,R² test,CV RMSE,Beats baseline (%)
SLR – Model A,1,0.0780,0.0761,41.2658,0.0000,10.4403,11.5839,0.0841,10.4268,4.3007
SLR – Model B,1,0.0348,0.0328,17.6033,0.0000,10.6818,11.9162,0.0308,10.6891,1.5549
MLR – Full,54,0.4955,0.4329,7.9126,0.0000,7.7226,8.8445,0.4661,8.8936,26.9317
MLR – Final (Forward (BIC)),10,0.4093,0.3969,33.1876,0.0000,8.3566,8.7487,0.4776,8.6289,27.7233


Naive baseline (training mean for every crash): RMSE test = 12.10 mph


## 6. Discussion

In [31]:
# @title
from IPython.display import Markdown

a_row, b_row, full_r, f_row = (final_table.iloc[i] for i in range(4))
Markdown(f"""
| | Variables | R² train | Adj. R² | RMSE train | RMSE test | R² test |
|---|---|---|---|---|---|---|
| Model A (SLR) | 1 | {a_row['R² train']:.3f} | {a_row['Adj. R²']:.3f} | {a_row['RMSE train']:.2f} | {a_row['RMSE test']:.2f} | {a_row['R² test']:.3f} |
| Model B (SLR) | 1 | {b_row['R² train']:.3f} | {b_row['Adj. R²']:.3f} | {b_row['RMSE train']:.2f} | {b_row['RMSE test']:.2f} | {b_row['R² test']:.3f} |
| Full MLR | {int(full_r['Variables'])} | {full_r['R² train']:.3f} | {full_r['Adj. R²']:.3f} | {full_r['RMSE train']:.2f} | {full_r['RMSE test']:.2f} | {full_r['R² test']:.3f} |
| **Final MLR** | **{int(f_row['Variables'])}** | **{f_row['R² train']:.3f}** | **{f_row['Adj. R²']:.3f}** | **{f_row['RMSE train']:.2f}** | **{f_row['RMSE test']:.2f}** | **{f_row['R² test']:.3f}** |
""")


| | Variables | R² train | Adj. R² | RMSE train | RMSE test | R² test |
|---|---|---|---|---|---|---|
| Model A (SLR) | 1 | 0.078 | 0.076 | 10.44 | 11.58 | 0.084 |
| Model B (SLR) | 1 | 0.035 | 0.033 | 10.68 | 11.92 | 0.031 |
| Full MLR | 54 | 0.496 | 0.433 | 7.72 | 8.84 | 0.466 |
| **Final MLR** | **10** | **0.409** | **0.397** | **8.36** | **8.75** | **0.478** |


**1. Final model.** 11 predictors chosen by forward BIC. It explains **47%** of the variation in speed (adjusted R² 0.46), far more than either simple regression (best adjusted R² 0.09).

**2. What drives speed.** Road type and what the other vehicle was doing matter most: highway crashes are about 32 mph faster, parking-lot crashes about 7 mph slower.

**3. Does it generalise?** Test RMSE is **8.76 mph** vs 8.33 on training – a small gap, so no serious overfitting. It is **11%** more accurate than predicting the average speed.

**4. Limits.** 53% of the variation is unexplained. Some predictors (police investigation, side impact) are recorded after the crash, so the model describes crashes rather than predicting them. Highway crashes are rare (9 in training), so that coefficient is less certain.